# Úkol 3c: Structured Outputs nad DataCorp daty

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shippy/czechitas-ai-data/blob/main/notebooks/assignment-03c.ipynb)

Pokud nebudete vědět, nezapomeňte se zeptat [pomocníčka pro tento úkol](https://chatgpt.com/g/g-67cab661ae5c8191b0d8419c76d3959b-czechitas-ai-in-data-analytics-2025-03)!

## Kontext

V tomto notebooku použijeme structured outputs k analýze nestrukturovaných textových dat z DataCorpu:
- **Performance reviews** (`datacorp_reviews.csv`) — hodnocení zaměstnanců jejich nadřízenými
- **Exit interviews** (`datacorp_exit_interviews.csv`) — rozhovory s odcházejícími zaměstnanci

Cílem je vytáhnout z volného textu strukturované informace, které pak můžeme propojit s hlavním datasetem `datacorp.csv`.

## Setup

Spusťte následující dvě buňky. Pokud vám AI řekne, jak se má, vše funguje správně.

In [ ]:
try:
    from google.colab import userdata
    _secret = userdata.get("OPENAI_API_KEY")
    %pip install pandas instructor openai python-dotenv rich seaborn
    !curl -sO https://raw.githubusercontent.com/shippy/czechitas-ai-data/refs/heads/main/notebooks/datacorp.csv
    !curl -sO https://raw.githubusercontent.com/shippy/czechitas-ai-data/refs/heads/main/notebooks/datacorp_reviews.csv
    !curl -sO https://raw.githubusercontent.com/shippy/czechitas-ai-data/refs/heads/main/notebooks/datacorp_exit_interviews.csv
except ImportError:
    import os
    from dotenv import load_dotenv
    _ = load_dotenv()
    _secret = os.environ.get("OPENAI_API_KEY")

In [ ]:
import asyncio
import instructor
from openai import OpenAI, AsyncOpenAI
from rich import print

try:
    if not _secret:
        raise ValueError("API klíč nebyl nastaven!")
except NameError:
    print("Nastavte si API klíč v proměnné OPENAI_API_KEY a znovu spusťte *celý* notebook včetně předchozí buňky")

# Synchronní klient — pro test a jednoduché volání
_client = OpenAI(api_key=_secret)
client = instructor.from_openai(_client)

# Asynchronní klient — pro paralelní zpracování mnoha řádků
_async_client = AsyncOpenAI(api_key=_secret)
async_client = instructor.from_openai(_async_client)

# Semaphore omezuje počet souběžných požadavků (abychom nepřetížili API)
SEM = asyncio.Semaphore(5)

test = client.chat.completions.create(
    messages=[
        {"role": "user", "content": "Hello, how are you? Respond with an emotion."},
    ],
    model="gpt-5-main-mini",
    response_model=str,
)
print(test)

## Čtení dat

In [ ]:
import pandas as pd

datacorp = pd.read_csv("datacorp.csv")
reviews = pd.read_csv("datacorp_reviews.csv")
exit_interviews = pd.read_csv("datacorp_exit_interviews.csv")

print(f"Zaměstnanci: {datacorp.shape[0]} řádků, Reviews: {reviews.shape[0]} řádků, Exit interviews: {exit_interviews.shape[0]} řádků")

In [ ]:
reviews.head()

In [ ]:
exit_interviews.head()

## Ukázka: Strukturovaný výstup z exit interview

Podívejme se, jak z volného textu exit interview vytáhnout strukturované informace:

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ExitReason(BaseModel):
    primary_reason: Literal["plat", "vedení", "kariérní růst", "kultura", "jiná nabídka", "osobní důvody", "jiné"]
    sentiment: Literal["pozitivní", "neutrální", "negativní"]
    summary: str = Field(..., description="Shrnutí v jedné větě")

async def extract_exit_reason(text: str) -> ExitReason:
    async with SEM:
        return await async_client.chat.completions.create(
            messages=[
                {"role": "system", "content": "Analyzuj text z exit interview zaměstnance. Extrahuj hlavní důvod odchodu, sentiment a shrnutí."},
                {"role": "user", "content": text},
            ],
            model="gpt-5-main-mini",
            response_model=ExitReason,
        )

# Ukázka na prvním záznamu
example = exit_interviews.iloc[0]
print(f"Text: {example['interview_text']}\n")

result = await extract_exit_reason(example["interview_text"])
print(result)

## Úkol 1: Analýza exit interviews

Zpracujte **všechny** exit interviews pomocí structured outputs. Pro každý záznam extrahujte:
- hlavní důvod odchodu
- sentiment (pozitivní / neutrální / negativní)
- jednověté shrnutí

Výsledky uložte do nového sloupce `exit_interviews['structured']`.

In [ ]:
results = await asyncio.gather(*[
    extract_exit_reason(row["interview_text"])
    for _, row in exit_interviews.iterrows()
])

exit_interviews["structured"] = results
print(f"Zpracováno {len(results)} exit interviews")
exit_interviews[["employee_id", "structured"]].head()

## Úkol 2: Analýza performance reviews

Nadefinujte vlastní Pydantic model pro performance reviews a zpracujte všechny záznamy. Zamyslete se, jaké informace lze z review textu vytáhnout — např.:
- celkový sentiment hodnocení
- silné stránky zaměstnance
- oblasti ke zlepšení
- doporučená akce (povýšení, rozvojový plán, varování, …)

Tip: podívejte se na několik review textů a na základě nich navrhněte strukturu.

In [ ]:
# Nadefinujte si vlastní Pydantic model
class ReviewAnalysis(BaseModel):
    ...  # doplňte atributy

async def extract_review(text: str) -> ReviewAnalysis:
    async with SEM:
        return await async_client.chat.completions.create(
            messages=[
                {"role": "system", "content": "..."},  # doplňte system prompt
                {"role": "user", "content": text},
            ],
            model="gpt-5-main-mini",
            response_model=ReviewAnalysis,
        )

review_results = await asyncio.gather(*[
    extract_review(row["review_text"])
    for _, row in reviews.iterrows()
])

# reviews['structured'] = review_results

## Úkol 3: Propojení s hlavním datasetem

Propojte výsledky z úkolů 1 a 2 s hlavním datasetem `datacorp.csv` přes sloupec `employee_id`. Pak zodpovězte:

1. **Které oddělení má nejvíc negativních exit interviews?**
2. **Koreluje sentiment review s výší platu nebo hodnocením výkonu?**
3. **Jaké jsou nejčastější důvody odchodu?** Vytvořte graf.

Tip: Vytvořte si pomocné sloupce z extrahovaných dat (např. `exit_interviews['primary_reason'] = [r.primary_reason for r in results]`).

In [ ]:
# Zde řešte úkol 3. (Můžete si vytvořit i více buněk!)

## Bonusový úkol: Vylepšete svůj model

- Zkuste přidat do `ExitReason` nebo `ReviewAnalysis` další pole — např. `konkrétní_stížnosti: list[str]` nebo `doporučení_pro_firmu: str`.
- Zkuste změnit system prompt a porovnejte, jak se změní výsledky.
- Zkuste použít `Field(description=...)` pro přesnější výstupy.

In [ ]:
# Zde řešte bonusový úkol.